# 第 25 课：模型导出——ONNX、固定接口与运行时一致性

训练代码不是部署接口。本课把一个带显式 cache 的因果声学模型导出为 ONNX，并逐项验证 PyTorch 与 ONNX Runtime 输出。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 量化与部署 |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 24 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | ONNX 导出、显式 cache、运行时一致性 |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：ONNX 导出、显式 cache、运行时一致性。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：PyTorch 基线输出；shape/dtype/dynamic axes；流式 cache；P50/P99 与 RTF。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [上一课](24_流式CTC系统综合实验与验收.ipynb)与[唯一学习路径](../LEARNING_PATH.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：冻结模型与数值基线、目标硬件、输入输出/状态协议
  ↓ 本课要学会的变换、状态或判断
输出：有一致性、性能、并发隔离和回滚证据的部署产物
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [ ]:
from pathlib import Path
import time
import sys
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/"pyproject.toml").exists(): return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT=find_root(); ARTIFACTS=ROOT/"artifacts";ARTIFACTS.mkdir(exist_ok=True)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
plt.rcParams["figure.figsize"]=(11,4)
print("项目根目录:",ROOT)

import onnx
import onnxruntime as ort
import torch
from deployment.model import (build_model,export_onnx,compare_torch_onnx,
    FEATURE_DIM,NUM_CLASSES,CHUNK_FRAMES,CACHE_FRAMES)

## 1. 先固定模型 contract

```text
frames:    float32 [1, 8, 24]
cache:     float32 [1, 2, 24]
logits:    float32 [1, 8, 11]
new_cache: float32 [1, 2, 24]
```

固定 chunk shape 通常更容易优化和部署；代价是客户端必须缓冲并处理最后不足一块的数据。

In [ ]:
model=build_model()
x=torch.randn(1,CHUNK_FRAMES,FEATURE_DIM);cache=torch.zeros(1,CACHE_FRAMES,FEATURE_DIM)
with torch.inference_mode(): logits,new_cache=model(x,cache)
print("frames",x.shape,"cache",cache.shape,"logits",logits.shape,"new_cache",new_cache.shape)

## 2. 使用新的 `torch.onnx.export(..., dynamo=True)` 导出

In [ ]:
onnx_path=export_onnx(ARTIFACTS/"streaming_ctc_demo.onnx")
print("saved:",onnx_path,"size KB:",onnx_path.stat().st_size/1024)
model_proto=onnx.load(onnx_path);onnx.checker.check_model(model_proto)
print("opset:",[(x.domain,x.version) for x in model_proto.opset_import])
print("nodes:",[n.op_type for n in model_proto.graph.node])

## 3. 一致性不是“能加载就算成功”

至少比较多组输入的最大绝对误差、相对误差、余弦相似度和最终解码结果。下面先做单组逐张量比较。

In [ ]:
print(compare_torch_onnx(onnx_path))
session=ort.InferenceSession(str(onnx_path),providers=["CPUExecutionProvider"])
print("inputs:",[(i.name,i.shape,i.type) for i in session.get_inputs()])
print("outputs:",[(o.name,o.shape,o.type) for o in session.get_outputs()])

## 4. 多 chunk cache 一致性

In [ ]:
rng=np.random.default_rng(8); chunks=rng.normal(size=(6,1,CHUNK_FRAMES,FEATURE_DIM)).astype(np.float32)
torch_cache=torch.zeros(1,CACHE_FRAMES,FEATURE_DIM);ort_cache=np.zeros((1,CACHE_FRAMES,FEATURE_DIM),np.float32)
errors=[]
for chunk in chunks:
    with torch.inference_mode(): tlogits,torch_cache=model(torch.from_numpy(chunk),torch_cache)
    ologits,ort_cache=session.run(None,{"frames":chunk,"cache":ort_cache})
    errors.append(np.max(np.abs(tlogits.numpy()-ologits)))
print("per-chunk max errors:",errors)

## 5. 导出常见坑

- 训练态没有切换 `eval()`；
- 把 Python 控制流或无法导出的算子带进图；
- cache shape/顺序不一致；
- 忘记 opset、输入名称、dtype；
- 只验一组随机输入；
- 导出成功但目标 runtime 不支持某个算子。

## 本课测试

1. ONNX 是训练框架还是模型交换表示？
2. 为什么 cache 必须成为显式输入输出？
3. 固定 chunk shape 的优缺点是什么？
4. 模型能被 ORT 加载是否等于结果正确？
5. 为什么要做连续多 chunk 一致性测试？

<details><summary>展开参考答案</summary>

1. 模型交换/计算图表示。2. 服务跨请求保存状态，runtime 不知道 Python 对象内部状态。3. 易优化但客户端需严格分块和补齐。4. 不等于。5. cache 错误可能第一块看不出来，之后逐步累积。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 25 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `ONNX 导出`、`显式 cache`、`运行时一致性`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**第一块一致但连续 cache 逐块漂移**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**导出后用多组多 chunk 比较 PyTorch/ORT**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**为 INT8 量化建立 FP32 基线**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：ONNX 导出、显式 cache、运行时一致性。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 ONNX 导出、显式 cache、运行时一致性。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
